### Notebook para perfilamiento de la base de datos.

In [0]:
# -*- coding: utf-8 -*-
"""
Perfilamiento de datos ADRES/MIPRES con Spark + Pandas (SELECT * global)
- 100% autocontenido (no lee archivos externos).
- Consulta global SELECT *.
- Guarda la tabla COMPLETA con Spark (Parquet en dbfs:/).
- Convierte a Pandas de forma SEGURA (muestreo si no cabe en RAM) para perfilar.
- Exporta perfiles (CSV/JSON) y muestra interpretación básica (y específica si detecta columnas clave).
"""

# ╭────────────────────────── Imports ──────────────────────────╮
import os
import json
import math
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List, Optional
from pyspark.sql import SparkSession

# ╭───────────── 0) Configuración base y utilidades ───────────╮
# Carpeta "humana" (solo para dejar archivos chicos como perfiles CSV/JSON)
BASE_DIR = "/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas"
CARPETA_SALIDA = os.path.join(BASE_DIR, "data/raw")
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Ruta RECOMENDADA para la tabla COMPLETA (Spark escribe en almacenamiento distribuido)
SPARK_PARQUET_DIR = "dbfs:/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/data/raw/tabla_completa_parquet"

# Archivo Parquet (versión Pandas: puede ser muestra o full si se lo forzas)
PANDAS_PARQUET_PATH = os.path.join(CARPETA_SALIDA, "tabla_pandas.parquet")

# Control de memoria para Pandas
FORZAR_PANDAS_COMPLETO = False   #  True para intentar traer todo a Pandas
MEMORIA_MAX_GB_PANDAS   = 3.0    # Límite blando de RAM para estimar muestra
MUESTRA_FALLBACK_FILAS  = 200000 # Si no se puede estimar bien,se intenta como tope de filas

CONFIG_ECHO = {
  "use_database": True,
  "db_type": "spark",
  "catalog": "mipres_catalog",
  "output_folder": "mlops_canvas/data/raw",
  "spark_parquet_dir": SPARK_PARQUET_DIR,
  "pandas_parquet_path": PANDAS_PARQUET_PATH
}

def bytes_to_human(nbytes: int) -> str:
    if not isinstance(nbytes, (int, float)) or nbytes <= 0:
        return "0 B"
    units = ["B","KB","MB","GB","TB","PB"]
    power = int(math.floor(math.log(nbytes, 1024)))
    power = min(power, len(units)-1)
    value = nbytes / (1024 ** power)
    return f"{value:.2f} {units[power]}"

def infer_logical_type(s: pd.Series) -> str:
    if pd.api.types.is_datetime64_any_dtype(s):
        return "datetime"
    if pd.api.types.is_bool_dtype(s):
        return "boolean"
    if pd.api.types.is_numeric_dtype(s):
        return "numeric"
    return "categorical"

# ╭───────────── 1) Definir la consulta SQL (GLOBAL: SELECT *) ─╮
consulta_sql = """
SELECT *
FROM mipres_catalog.bronze_db_mipres_suministro.dbo_tsum_tx
-- WHERE FechaRegistro BETWEEN '2024-01-01' AND current_date()  -- (opcional)
"""

# ╭───────────── 2) Ejecutar la consulta en Spark ─────╮
def ejecutar_sql_en_spark(sql_text: str, catalog: str = "mipres_catalog"):
    spark = SparkSession.builder.getOrCreate()
    try:
        if catalog:
            spark.sql(f"USE CATALOG {catalog}")
    except Exception as e:
        print(f"[Aviso] No se pudo usar el catálogo '{catalog}': {e}. Se continúa sin USE CATALOG.")
    return spark.sql(sql_text)

# ╭───────────── 3) Convertir el resultado a Pandas (seguro) ─╮
def estimar_bytes_por_fila(df_spark, muestra_n: int = 10000) -> Optional[float]:
    try:
        pdf = df_spark.limit(muestra_n).toPandas()  # toPandas trae datos al driver; úsese con muestras
        if len(pdf) == 0:
            return None
        return float(pdf.memory_usage(deep=True).sum()) / float(len(pdf))
    except Exception as e:
        print(f"[Aviso] No se pudo estimar bytes/fila por muestra: {e}")
        return None

def colectar_pandas_seguro(df_spark,
                           forzar_full: bool = False,
                           memoria_max_gb: float = 3.0,
                           fallback_filas: int = 200000,
                           seed: int = 42) -> Tuple[pd.DataFrame, str]:
    """
    Devuelve (df_pandas, nota_origen), donde nota_origen ∈ {"full","sample"}.
    Si la estimación de memoria supera 'memoria_max_gb', toma una muestra.
    """
    spark = SparkSession.builder.getOrCreate()
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

    total = df_spark.count()
    if total == 0:
        return pd.DataFrame(), "full"

    if forzar_full:
        print("[Aviso] FORZAR_PANDAS_COMPLETO=True. Intentando traer toda la tabla a Pandas (bajo tu responsabilidad).")
        return df_spark.toPandas(), "full"

    # Estimar bytes por fila con una muestra chica
    bpf = estimar_bytes_por_fila(df_spark, muestra_n=10000)
    if bpf is None or bpf <= 0:
        # Fallback: limitar filas
        n_target = min(total, fallback_filas)
        print(f"[Aviso] No se pudo estimar memoria. Colectando fallback de {n_target} filas con limit().")
        return df_spark.limit(n_target).toPandas(), "sample"

    memoria_total_est = bpf * total
    cap_bytes = memoria_max_gb * (1024**3)
    print(f"[Estimación] Filas={total:,} | B/fila≈{bpf:,.2f} | Estimado total≈{bytes_to_human(memoria_total_est)} | Cap≈{memoria_max_gb} GB")

    if memoria_total_est <= cap_bytes:
        return df_spark.toPandas(), "full"

    # Calcular fracción de muestreo para cumplir cap
    target_rows = max(int(cap_bytes // bpf), 1)
    frac = min(max(target_rows / total, 1e-5), 1.0)
    print(f"[Muestreo] Usando fraction≈{frac:.6f} (~{target_rows:,} filas).")
    sampled = df_spark.sample(withReplacement=False, fraction=frac, seed=seed)
    return sampled.toPandas(), "sample"

# ╭───────────── 4) Guardar la tabla en Parquet ───────────────╮
def guardar_full_con_spark(df_spark, path_dbfs: str) -> None:
    """
    Escribe la TABLA COMPLETA con Spark en Parquet (distribuido).
    Ruta tipo 'dbfs:/...' (recomendado para grandes volúmenes). 
    Documentación DataFrameWriter.parquet.  # ver cita
    """
    (df_spark.write
      .mode("overwrite")
      .parquet(path_dbfs))
    print(f"[4A] Tabla COMPLETA guardada con Spark en: {path_dbfs}")

def guardar_pandas_parquet(df_pandas: pd.DataFrame, ruta_parquet: str) -> None:
    df_pandas.to_parquet(ruta_parquet, index=False)
    print(f"[4B] Pandas Parquet guardado en: {ruta_parquet}")

# ╭───────────── 5) Verificar lectura del Parquet (Pandas) ───╮
def verificar_parquet_pandas(ruta_parquet: str, head_n: int = 5) -> pd.DataFrame:
    dfv = pd.read_parquet(ruta_parquet)
    print(f"[5] Verificación Pandas Head({head_n}):")
    print(dfv.head(head_n))
    return dfv

# ╭───────────── 6) Perfilamiento de la base (Pandas) ────────╮
def perfilar_dataframe(df: pd.DataFrame) -> Tuple[Dict, pd.DataFrame, pd.DataFrame]:
    n_rows, n_cols = df.shape
    mem_bytes = int(df.memory_usage(deep=True).sum()) if n_rows > 0 else 0
    n_dupes = int(df.duplicated().sum()) if n_rows > 0 else 0

    per_col_records: List[Dict] = []
    for col in df.columns:
        s = df[col]
        logical = infer_logical_type(s)
        dtype = str(s.dtype)

        n_missing = int(s.isna().sum()) if n_rows > 0 else 0
        pct_missing = (n_missing / n_rows * 100.0) if n_rows > 0 else 0.0

        n_unique = int(s.nunique(dropna=True)) if n_rows > 0 else 0
        pct_unique = (n_unique / n_rows * 100.0) if n_rows > 0 else 0.0

        constant = (n_unique == 1 and n_rows > 0)
        zeros = negs = None
        min_val = max_val = mean = std = q1 = q3 = median = skew = kurt = None
        mindate = maxdate = None

        if logical == "numeric":
            zeros = int((s == 0).sum())
            negs = int((s < 0).sum())
            desc = s.describe(percentiles=[0.25, 0.5, 0.75])
            min_val  = float(desc.get("min", np.nan)) if "min" in desc else None
            max_val  = float(desc.get("max", np.nan)) if "max" in desc else None
            mean     = float(desc.get("mean", np.nan)) if "mean" in desc else None
            std      = float(desc.get("std", np.nan)) if "std" in desc else None
            q1       = float(desc.get("25%", np.nan)) if "25%" in desc else None
            median   = float(desc.get("50%", np.nan)) if "50%" in desc else None
            q3       = float(desc.get("75%", np.nan)) if "75%" in desc else None
            skew     = float(s.skew(skipna=True)) if n_rows > 1 else None
            kurt     = float(s.kurtosis(skipna=True)) if n_rows > 1 else None

        elif logical == "datetime":
            s_dt = pd.to_datetime(s, errors="coerce")
            mindate = s_dt.min()
            maxdate = s_dt.max()

        top_values = None
        if n_unique <= 20 and n_rows > 0:
            vc = s.value_counts(dropna=True).head(5)
            top_values = "; ".join([f"{idx}:{cnt}" for idx, cnt in vc.items()])

        per_col_records.append({
            "columna": col,
            "dtype_pandas": dtype,
            "tipo_logico": logical,
            "n_missing": n_missing,
            "pct_missing": round(pct_missing, 4),
            "n_unicos": n_unique,
            "pct_unicos": round(pct_unique, 4),
            "constante": constant,
            "ceros": zeros,
            "negativos": negs,
            "min": min_val,
            "q1": q1,
            "mediana": median,
            "q3": q3,
            "max": max_val,
            "media": mean,
            "desv_std": std,
            "asimetria": skew,
            "curtosis": kurt,
            "fecha_min": str(mindate) if mindate is not None and not pd.isna(mindate) else None,
            "fecha_max": str(maxdate) if maxdate is not None and not pd.isna(maxdate) else None,
            "top_valores": top_values
        })

    per_col_df = pd.DataFrame(per_col_records)
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    corr_df = df[numeric_cols].corr(method="pearson") if numeric_cols else pd.DataFrame()

    summary = {
        "n_filas_pandas": int(n_rows),
        "n_columnas": int(n_cols),
        "memoria_bytes_pandas": mem_bytes,
        "memoria_humana_pandas": bytes_to_human(mem_bytes),
        "n_duplicados": n_dupes,
        "columnas": df.columns.tolist(),
        "tipos": {c: str(df[c].dtype) for c in df.columns}
    }
    return summary, per_col_df, corr_df

# ╭───────────── 7) Guardar reportes de perfil ───────╮
def exportar_perfiles(per_col_df: pd.DataFrame, corr_df: pd.DataFrame, out_dir: str) -> Tuple[str, str]:
    per_col_path = os.path.join(out_dir, "perfil_columnas.csv")
    corr_path = os.path.join(out_dir, "correlaciones.csv")
    per_col_df.to_csv(per_col_path, index=False, encoding="utf-8")
    if not corr_df.empty:
        corr_df.to_csv(corr_path, index=True, encoding="utf-8")
    else:
        pd.DataFrame().to_csv(corr_path, index=False)
    return per_col_path, corr_path

def exportar_resumen_json(summary: Dict, out_dir: str) -> str:
    summary_path = os.path.join(out_dir, "resumen_global.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    return summary_path

# ╭───────────── 8) Interpretación (genérica + específica ADRES) ─╮
def interpretar_resultado(df: pd.DataFrame) -> None:
    print("\n[8] Interpretación (rápida):")
    if df.empty:
        print("- El DataFrame Pandas está vacío (puede que hayas guardado solo con Spark).")
        print("- Perfilamiento Pandas no disponible. Crea una muestra incrementando MEMORIA_MAX_GB_PANDAS o bajando filtros.")
        return

    # Genérica:
    print(f"- Observadas {len(df):,} filas (en Pandas; podría ser muestra).")
    # Específica si detecta columnas habituales:
    cols = set(map(str.lower, df.columns))
    candidatos_fecha = [c for c in df.columns if 'fecha' in c.lower()]
    if candidatos_fecha:
        for c in candidatos_fecha[:3]:
            s = pd.to_datetime(df[c], errors="coerce")
            print(f"  · Rango {c}: {str(s.min())} → {str(s.max())}")
    # Usuarios únicos si existen columnas tipo identificación
    posibles_id = [c for c in df.columns if c.lower() in ("tipoidpaciente","noidpaciente")]
    if len(posibles_id) == 2:
        tuplas = df[posibles_id].dropna().drop_duplicates().shape[0]
        print(f"- Personas únicas (aprox. sobre Pandas): {tuplas:,} (por {posibles_id})")
    if "codigoepsprescripcion" in cols:
        top_eps = df["CodigoEPSPrescripcion"].value_counts(dropna=True).head(5)
        print("- Top 5 EPS por frecuencia (Pandas, aprox.):")
        print(top_eps)

# ╭───────────── Main: orquestación ─────────────╮
if __name__ == "__main__":
    print("[Config] Parámetros de ejecución:")
    print(json.dumps(CONFIG_ECHO, ensure_ascii=False, indent=2))

    # 2) Ejecutar SQL (SELECT * global)
    df_spark = ejecutar_sql_en_spark(consulta_sql, catalog=CONFIG_ECHO.get("catalog"))
    print("[2] Consulta ejecutada en Spark (SELECT *).")
    print(f"    - Columnas Spark: {len(df_spark.columns)}")
    try:
        total_rows = df_spark.count()
        print(f"    - Filas (Spark): {total_rows:,}")
    except Exception as e:
        print(f"    - No se pudo contar filas Spark: {e}")

    # 3) Spark ➜ Pandas (seguro)
    df_pandas, origen = colectar_pandas_seguro(df_spark,
                                               forzar_full=FORZAR_PANDAS_COMPLETO,
                                               memoria_max_gb=MEMORIA_MAX_GB_PANDAS,
                                               fallback_filas=MUESTRA_FALLBACK_FILAS)
    print(f"[3] Convertido a Pandas ({origen}). shape = {df_pandas.shape}")

    # 4A) Guardar TABLA COMPLETA con Spark (Parquet distribuido en dbfs:/)
    guardar_full_con_spark(df_spark, SPARK_PARQUET_DIR)

    # 4B) Guardar versión Pandas (muestra o full, según 'origen')
    guardar_pandas_parquet(df_pandas, PANDAS_PARQUET_PATH)

    # 5) Verificar lectura (Pandas)
    try:
        df_verificacion = verificar_parquet_pandas(PANDAS_PARQUET_PATH, head_n=5)
    except Exception as e:
        print(f"[5] No se pudo verificar Parquet Pandas: {e}")
        df_verificacion = df_pandas

    # 6) Perfilamiento (Pandas)
    summary, per_col_df, corr_df = perfilar_dataframe(df_verificacion)
    print("\n[6] Resumen global (Pandas):")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    # 7) Exportes de perfil
    per_col_path, corr_path = exportar_perfiles(per_col_df, corr_df, CARPETA_SALIDA)
    summary_path = exportar_resumen_json(summary, CARPETA_SALIDA)
    print(f"\n[7] Reportes exportados:\n- {per_col_path}\n- {corr_path}\n- {summary_path}")

    # 8) Interpretación
    interpretar_resultado(df_verificacion)

    print("\n✅ Proceso completo.")


In [0]:
    # 1) Ruta al archivo perfil_columnas.csv — ajusta si está en otro directorio
    archivo = "/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/data/raw/perfil_columnas.csv"
    if not os.path.exists(archivo):
        print(f"Error: El archivo no existe en la ruta: {archivo}")
        pass

    # 2) Leer con pandas
    df = pd.read_csv(archivo, encoding="utf-8")
    print("Archivo perfil_columnas.csv cargado correctamente.")
    
    # 3) Ver primeras filas
    print("\nPrimeras filas:")
    display(df)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
try:
    spark.sql("USE CATALOG mipres_catalog")
except Exception as e:
    print(f"[Aviso] No se pudo usar catálogo: {e}")

BASE_TBL = "mipres_catalog.bronze_db_mipres_suministro.dbo_tsum_tx"

# Total de EPS distintas en todo el dataset
df_eps_distintas = spark.sql(f"""
SELECT COUNT(DISTINCT CodigoEPSPrescripcion) AS eps_distintas
FROM {BASE_TBL}
""")
try:
    display(df_eps_distintas)
except NameError:
    df_eps_distintas.show(truncate=False)

# Un top de EPS por # de registros (para tener contexto visual)
df_top_eps = spark.sql(f"""
SELECT CodigoEPSPrescripcion, COUNT(*) AS registros
FROM {BASE_TBL}
GROUP BY CodigoEPSPrescripcion
ORDER BY registros DESC
LIMIT 50
""")
try:
    display(df_top_eps)
except NameError:
    df_top_eps.show(50, truncate=False)


In [0]:
# -*- coding: utf-8 -*-
# Notebook: /Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/notebooks/database

from pathlib import Path
from pyspark.sql import SparkSession
import shutil

# ───────────────── 0) Rutas locale  ─────────────────
PROJECT_ROOT = Path("/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas")
OUT_DIR      = PROJECT_ROOT / "data" / "raw" / "complete"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "df.parquet"          # archivo final único
TMP_DIR  = OUT_DIR / "_tmp_parquet"         # carpeta temporal local

# ───────────────── 1) Spark ─────────────────
spark = SparkSession.builder.getOrCreate()
try:
    spark.sql("USE CATALOG mipres_catalog")
except Exception as e:
    print(f"[Aviso] No se pudo usar catálogo: {e}. Seguimos.")

BASE_TBL = "mipres_catalog.bronze_db_mipres_suministro.dbo_tsum_tx"

# ───────────────── 2) SQL ─────────────────
sql = f"""
WITH base_w AS (
  SELECT
    date_trunc('week', FechaRegistro) AS semana,
    FechaRegistro,
    TipoTecnologia,
    RegimenPrescripcion,
    CodigoETPrescripcion,
    CodigoEPSPrescripcion,
    struct(TipoIDPaciente, NoIDPaciente) AS id_paciente
  FROM {BASE_TBL}
),
marcas AS (
  SELECT
    b.*,
    MIN(semana) OVER (PARTITION BY id_paciente) AS primera_semana_global
  FROM base_w b
),
sem_core AS (
  SELECT
    semana,
    COUNT(*)                                        AS total_suministros_semana,
    COUNT(DISTINCT id_paciente)                     AS total_pacientes_unicos_semana,
    SUM(CASE WHEN TipoTecnologia='M' THEN 1 ELSE 0 END) AS n_tipo_m,
    SUM(CASE WHEN TipoTecnologia='P' THEN 1 ELSE 0 END) AS n_tipo_p,
    SUM(CASE WHEN TipoTecnologia='D' THEN 1 ELSE 0 END) AS n_tipo_d,
    SUM(CASE WHEN TipoTecnologia='N' THEN 1 ELSE 0 END) AS n_tipo_n,
    SUM(CASE WHEN TipoTecnologia='S' THEN 1 ELSE 0 END) AS n_tipo_s,
    SUM(CASE WHEN RegimenPrescripcion='C' THEN 1 ELSE 0 END) AS n_reg_c,
    SUM(CASE WHEN RegimenPrescripcion='S' THEN 1 ELSE 0 END) AS n_reg_s,
    SUM(CASE WHEN RegimenPrescripcion IS NULL OR RegimenPrescripcion NOT IN ('C','S') THEN 1 ELSE 0 END) AS n_reg_otro
  FROM base_w
  GROUP BY semana
),
hhi_w AS (
  SELECT
    semana,
    SUM( POW(depto_cnt / total_semana, 2) ) AS hhi_et_semana
  FROM (
    SELECT
      semana,
      CodigoETPrescripcion,
      COUNT(*) AS depto_cnt,
      SUM(COUNT(*)) OVER (PARTITION BY semana) AS total_semana
    FROM base_w
    GROUP BY semana, CodigoETPrescripcion
  ) t
  GROUP BY semana
),
shares_w AS (
  SELECT
    c.semana,
    c.total_suministros_semana,
    c.total_pacientes_unicos_semana,
    (n_tipo_m / NULLIF(c.total_suministros_semana,0)) AS share_tipo_medicamento,
    (n_tipo_p / NULLIF(c.total_suministros_semana,0)) AS share_tipo_procedimiento,
    (n_tipo_d / NULLIF(c.total_suministros_semana,0)) AS share_tipo_dispositivo,
    (n_tipo_n / NULLIF(c.total_suministros_semana,0)) AS share_tipo_nutricional,
    (n_tipo_s / NULLIF(c.total_suministros_semana,0)) AS share_tipo_servicio_comp,
    (n_reg_c / NULLIF(c.total_suministros_semana,0))  AS share_regimen_contributivo,
    (n_reg_s / NULLIF(c.total_suministros_semana,0))  AS share_regimen_subsidiado,
    (n_reg_otro / NULLIF(c.total_suministros_semana,0)) AS share_regimen_otro
  FROM sem_core c
),
nuevos_pref AS (
  SELECT
    semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'EPS%%' THEN id_paciente END) AS u_nuevos_pref_eps_semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'EAS%%' THEN id_paciente END) AS u_nuevos_pref_eas_semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'ESS%%' THEN id_paciente END) AS u_nuevos_pref_ess_semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'CCF%%' THEN id_paciente END) AS u_nuevos_pref_ccf_semana
  FROM marcas
  GROUP BY semana
),
y_w AS (
  SELECT
    semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana THEN id_paciente END) AS y_usuarios_nuevos_semana
  FROM marcas
  GROUP BY semana
),
ensamblado AS (
  SELECT
    s.semana,
    y.y_usuarios_nuevos_semana,
    s.total_suministros_semana,
    s.total_pacientes_unicos_semana,
    s.share_tipo_medicamento,
    s.share_tipo_procedimiento,
    s.share_tipo_dispositivo,
    s.share_tipo_nutricional,
    s.share_tipo_servicio_comp,
    s.share_regimen_contributivo,
    s.share_regimen_subsidiado,
    s.share_regimen_otro,
    h.hhi_et_semana,
    p.u_nuevos_pref_eps_semana,
    p.u_nuevos_pref_eas_semana,
    p.u_nuevos_pref_ess_semana,
    p.u_nuevos_pref_ccf_semana
  FROM shares_w s
  LEFT JOIN hhi_w  h USING (semana)
  LEFT JOIN nuevos_pref p USING (semana)
  LEFT JOIN y_w y     USING (semana)
),
lags_t4_only AS (
  SELECT
    semana,
    y_usuarios_nuevos_semana,
    LAG(total_suministros_semana,      4) OVER (ORDER BY semana) AS total_suministros_t4,
    LAG(total_pacientes_unicos_semana, 4) OVER (ORDER BY semana) AS total_pacientes_unicos_t4,
    LAG(share_tipo_medicamento,        4) OVER (ORDER BY semana) AS share_tipo_medicamento_t4,
    LAG(share_tipo_procedimiento,      4) OVER (ORDER BY semana) AS share_tipo_procedimiento_t4,
    LAG(share_tipo_dispositivo,        4) OVER (ORDER BY semana) AS share_tipo_dispositivo_t4,
    LAG(share_tipo_nutricional,        4) OVER (ORDER BY semana) AS share_tipo_nutricional_t4,
    LAG(share_tipo_servicio_comp,      4) OVER (ORDER BY semana) AS share_tipo_servicio_comp_t4,
    LAG(share_regimen_contributivo,    4) OVER (ORDER BY semana) AS share_regimen_contributivo_t4,
    LAG(share_regimen_subsidiado,      4) OVER (ORDER BY semana) AS share_regimen_subsidiado_t4,
    LAG(share_regimen_otro,            4) OVER (ORDER BY semana) AS share_regimen_otro_t4,
    LAG(hhi_et_semana,                 4) OVER (ORDER BY semana) AS hhi_et_t4,
    LAG(u_nuevos_pref_eps_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_eps_t4,
    LAG(u_nuevos_pref_eas_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_eas_t4,
    LAG(u_nuevos_pref_ess_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_ess_t4,
    LAG(u_nuevos_pref_ccf_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_ccf_t4
  FROM ensamblado
),
final_ready AS (
  SELECT *
  FROM lags_t4_only
  WHERE total_suministros_t4 IS NOT NULL
)
SELECT *
FROM final_ready
ORDER BY semana
"""
df = spark.sql(sql)

# ───────────────── 3) Vista rápida (opcional) ─────────────────
try:
    display(df)
except Exception:
    df.show(20, truncate=False)

# ───────────────── 4) Exportar a archivo único Parquet en /Workspace ─────────────────
# Spark escribe en carpeta: usamos 'file:' para el filesystem local del driver,
# y luego movemos el único part-*.parquet a df.parquet.

# 4.1 limpiar tmp y escribir 1 solo part-file
if TMP_DIR.exists():
    shutil.rmtree(TMP_DIR)
# IMPORTANTE: usar esquema 'file:' para que Spark escriba en FS local (NO dbfs)
spark.write = df.coalesce(1).write.mode("overwrite")
spark.write.parquet(f"file:{TMP_DIR.as_posix()}")

# 4.2 localizar part-*.parquet y renombrar
parts = list(TMP_DIR.glob("part-*.parquet"))
if not parts:
    # ayuda de depuración
    print("[Depuración] Contenido de TMP_DIR:", TMP_DIR)
    for p in TMP_DIR.iterdir():
        print(" -", p)
    raise RuntimeError("No se encontró part-*.parquet en el directorio temporal local.")
part_file = parts[0]

# 4.3 reemplazar si existía un df.parquet previo
if OUT_FILE.exists():
    OUT_FILE.unlink()

# mover part -> df.parquet
part_file.replace(OUT_FILE)

# 4.4 limpiar temporal
shutil.rmtree(TMP_DIR, ignore_errors=True)

print(f"✅ Archivo Parquet único escrito en: {OUT_FILE}")


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
try:
    spark.sql("USE CATALOG mipres_catalog")
except Exception as e:
    print(f"[Aviso] No se pudo usar catálogo mipres_catalog: {e}")

# Listar tablas en la base de datos que usas, por ejemplo "bronze_db_mipres_suministro"
db_name = "bronze_db_mipres_suministro"

df_tables = spark.sql(f"SHOW TABLES IN mipres_catalog.{db_name}")
try:
    display(df_tables)
except NameError:
    df_tables.show(truncate=False)

# Se listan todas las bases de datos
df_databases = spark.sql("SHOW DATABASES IN mipres_catalog")
try:
    display(df_databases)
except NameError:
    df_databases.show(truncate=False)
